## Langfuse Adapter Demo 

This notebook demonstrates how to use the **EPR** and **WEPR** metrics at a sequence level with Langfuse to automatically score LLM outputs from traces.

### Prerequisites

**1. Langfuse project**

Create a free project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to **Settings → API Keys** and copy your keys.

Create a `.env.demo` file at the project root.

**2. Local LLM server**

This demo runs inference locally using `llama.cpp`. 

* Install ``llama.cpp`` using brew, nix or winget

Then start the server by running :
* ``llama-server -hf unsloth/SmolLM2-135M-Instruct-GGUF --port 8080``

For more info, see [github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)

### Run a generation and send it to Langfuse

We use the `@observe()` decorator to automatically trace the LLM call and send it to Langfuse.

In [4]:
from dotenv import load_dotenv
from langfuse.openai import OpenAI
from langfuse import observe, get_client

load_dotenv(".env.demo")

client = OpenAI(
    base_url="http://localhost:8080",
    api_key="test"
)

@observe()
def run_generation():
    completion = client.chat.completions.create(
        model="unsloth/SmolLM2-135M-Instruct-GGUF",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"},
        ],
        logprobs = True,
        top_logprobs = 5
    )
    return completion

completion = run_generation()
message = completion.choices[0].message.content
logprobs = completion.choices[0].logprobs

langfuse = get_client()
langfuse.flush()

### Score traces with EPR


In [5]:
from langfuse import get_client
from artefactual.adapters.langfuse_EPR import Artefactual_EPR
langfuse = get_client()

scorer = Artefactual_EPR(name="EPR", langfuse_client=langfuse)

traces = langfuse.api.trace.list(limit=2)

for trace in traces.data:
    score = scorer.score_trace(trace.id)
    print(f"EPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

EPR Scored Trace : 506030f83532ddf79c50533bf5f61f4b → 0.2944411635398865
EPR Scored Trace : c4abbeed6e9a757b9048dd533981d939 → 0.2944411635398865


### Score traces with WEPR

In [6]:
from langfuse import get_client
from artefactual.adapters.langfuse_WEPR import Artefactual_WEPR
langfuse = get_client()

scorer = Artefactual_WEPR(name="WEPR", langfuse_client=langfuse, pretrained_model_name_or_path="mistralai/Mistral-Small-3.1-24B-Instruct-2503")

traces = langfuse.api.trace.list(limit=2)

for trace in traces.data:
    score = scorer.score_trace(trace.id)
    print(f"WEPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

WEPR Scored Trace : 506030f83532ddf79c50533bf5f61f4b → 0.06835944205522537
WEPR Scored Trace : c4abbeed6e9a757b9048dd533981d939 → 0.06835944205522537
